In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

BASE = '../../new-dataset/tennis_MatchChartingProject-master'

matches   = pd.read_csv(f'{BASE}/charting-m-matches.csv')
overview  = pd.read_csv(f'{BASE}/charting-m-stats-Overview.csv')
rally     = pd.read_csv(f'{BASE}/charting-m-stats-Rally.csv')
kps  = pd.read_csv(f'{BASE}/charting-m-stats-KeyPointsServe.csv')
kpr = pd.read_csv(f'{BASE}/charting-m-stats-KeyPointsReturn.csv')

print('matches:   ', matches.shape)
print('overview:  ', overview.shape)
print('rally:     ', rally.shape)
print('kp_serve:  ', kps.shape)
print('kp_return: ', kpr.shape)

matches:    (7566, 15)
overview:   (56850, 20)
rally:      (96706, 13)
kp_serve:   (60464, 12)
kp_return:  (60464, 8)


### Objetivo

O objetivo é fazer um dataset final com o perfil de cada jogador até o dia do jogo e o resultado do confronto. Esse perfil vai ser feito a partir das tabelas dos jogos, rally, keypoints e o overview de cada partida já jogada em torneios oficiais de tênis. Ao final, vamos ter informações, por exemplo, das médias de vezes que ele retornou o primeiro saque, seu elo naquele momento, etc. Depois, transformaremos tudo em um dataset, com o perfil, confronto e o resultado.

Em seguida, iremos criar uma tabela final, com o perfil de todos os jogadores até aquele momento. Quando o usuário digitar um jogador vs o outro, vamos utilizar os perfis dessa tabela para fazer a predição.

Para fazer isso, precisamos entender os dados que temos e como podemos utilizá-los para fazer um resumo de cada jogador.



#### Modificando o dataset das partidas para incluir o perfil de cada jogador naquele dia

##### Convertendo a coluna Date para datetime

In [16]:
matches['Date'] = pd.to_datetime(matches['Date'], format='%Y%m%d', errors='coerce')
matches.columns

Index(['match_id', 'Player 1', 'Player 2', 'Pl 1 hand', 'Pl 2 hand', 'Date',
       'Tournament', 'Round', 'Time', 'Court', 'Surface', 'Umpire', 'Best of',
       'Final TB?', 'Charted by'],
      dtype='object')

##### Extraindo informações de overview

Extrair informações apenas do que aconteceu na partida como um todo, e não em cada set.

In [17]:
ov = overview.query("set == 'Total'").copy() 
ov.shape

(15116, 20)

In [18]:
ov['first_serve_pct']      = ov['first_in']       / ov['serve_pts']
ov['first_serve_won_pct']  = ov['first_won']       / ov['first_in'].replace(0, np.nan)
ov['second_serve_won_pct'] = ov['second_won']      / ov['second_in'].replace(0, np.nan)
ov['ace_pct']              = ov['aces']            / ov['serve_pts']
ov['df_pct']               = ov['dfs']             / ov['serve_pts']
ov['return_won_pct']       = ov['return_pts_won']  / ov['return_pts'].replace(0, np.nan)
ov['winners_per_pt']       = ov['winners']         / (ov['serve_pts'] + ov['return_pts'])
ov['ue_per_pt']            = ov['unforced']        / (ov['serve_pts'] + ov['return_pts'])
ov['winners_fh_ratio']     = ov['winners_fh']      / ov['winners'].replace(0, np.nan)
ov['bp_save_pct']          = ov['bp_saved']        / ov['bk_pts'].replace(0, np.nan)

In [19]:
ratio_cols = ['match_id', 'player', 'first_serve_pct', 'first_serve_won_pct',
              'second_serve_won_pct', 'ace_pct', 'df_pct', 'return_won_pct',
              'winners_per_pt', 'ue_per_pt', 'winners_fh_ratio', 'bp_save_pct']

ov_ratios = ov[ratio_cols]
print(ov_ratios.shape)
ov_ratios.head(4)

(15116, 12)


,match_id,player,first_serve_pct,first_serve_won_pct,second_serve_won_pct,ace_pct,df_pct,return_won_pct,winners_per_pt,ue_per_pt,winners_fh_ratio,bp_save_pct
0,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,Jesper De Jong,0.600000,0.687500,0.406250,0.075000,0.050000,0.295082,0.177305,0.177305,0.520000,0.727273
1,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,Michael Zheng,0.557377,0.794118,0.592593,0.049180,0.032787,0.425000,0.198582,0.148936,0.821429,1.000000
7,20260517-M-Rome_Masters-F-Casper_Ruud-Jannik_S...,Casper Ruud,0.578125,0.648649,0.481481,0.046875,0.031250,0.290909,0.184874,0.184874,0.590909,0.400000
8,20260517-M-Rome_Masters-F-Casper_Ruud-Jannik_S...,Jannik Sinner,0.636364,0.828571,0.500000,0.036364,0.000000,0.421875,0.226891,0.151261,0.555556,0.500000


In [20]:
ov_ratios = ov_ratios.merge(matches[['match_id', 'Date']], on='match_id', how='left')
ov_ratios['Date'] = pd.to_datetime(ov_ratios['Date'], format='%Y%m%d', errors='coerce')
print(ov_ratios.shape)
ov_ratios.head(3)


(15118, 13)


,match_id,player,first_serve_pct,first_serve_won_pct,second_serve_won_pct,ace_pct,df_pct,return_won_pct,winners_per_pt,ue_per_pt,winners_fh_ratio,bp_save_pct,Date
0,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,Jesper De Jong,0.600000,0.687500,0.406250,0.075000,0.050000,0.295082,0.177305,0.177305,0.520000,0.727273,2026-05-21
1,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,Michael Zheng,0.557377,0.794118,0.592593,0.049180,0.032787,0.425000,0.198582,0.148936,0.821429,1.000000,2026-05-21
2,20260517-M-Rome_Masters-F-Casper_Ruud-Jannik_S...,Casper Ruud,0.578125,0.648649,0.481481,0.046875,0.031250,0.290909,0.184874,0.184874,0.590909,0.400000,2026-05-17


In [21]:
ov_ratios = ov_ratios.sort_values(['player', 'Date'])

ratio_feature_cols = ['first_serve_pct', 'first_serve_won_pct', 'second_serve_won_pct',
                      'ace_pct', 'df_pct', 'return_won_pct', 'winners_per_pt',
                      'ue_per_pt', 'winners_fh_ratio', 'bp_save_pct']

for col in ratio_feature_cols:
    ov_ratios[f'avg_{col}'] = (
        ov_ratios.groupby('player')[col].transform(lambda x: x.expanding().mean().shift(1))
    )

print(ov_ratios.shape)
ov_ratios.head(10)


(15118, 23)


,match_id,player,first_serve_pct,first_serve_won_pct,second_serve_won_pct,ace_pct,df_pct,return_won_pct,winners_per_pt,ue_per_pt,...,avg_first_serve_pct,avg_first_serve_won_pct,avg_second_serve_won_pct,avg_ace_pct,avg_df_pct,avg_return_won_pct,avg_winners_per_pt,avg_ue_per_pt,avg_winners_fh_ratio,avg_bp_save_pct
14629,19890909-M-US_Open-SF-Boris_Becker-Aaron_Krick...,Aaron Krickstein,0.447917,0.720930,0.377358,0.062500,0.041667,0.414414,0.149758,0.149758,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14580,19900415-M-Tokyo_Outdoor-F-Aaron_Krickstein-St...,Aaron Krickstein,0.705128,0.618182,0.347826,0.038462,0.000000,0.388889,0.153333,0.133333,...,0.447917,0.720930,0.377358,0.062500,0.041667,0.414414,0.149758,0.149758,0.451613,0.400000
14348,19910902-M-US_Open-R16-Aaron_Krickstein-Jimmy_...,Aaron Krickstein,0.557214,0.660714,0.573034,0.034826,0.009950,0.385965,0.096774,0.139785,...,0.576522,0.669556,0.362592,0.050481,0.020833,0.401652,0.151546,0.141546,0.399719,0.543750
14322,19911026-M-Stockholm_Masters-SF-Aaron_Krickste...,Aaron Krickstein,0.571429,0.535714,0.380952,0.081633,0.000000,0.266667,0.095745,0.117021,...,0.570086,0.666609,0.432739,0.045262,0.017206,0.396423,0.133289,0.140959,0.396109,0.626389
14266,19920426-M-Monte_Carlo_Masters-F-Aaron_Krickst...,Aaron Krickstein,0.565789,0.465116,0.515152,0.026316,0.000000,0.310811,0.100000,0.226667,...,0.570422,0.633885,0.419793,0.054355,0.012904,0.363984,0.123903,0.134975,0.352638,0.541220
13970,19940305-M-Indian_Wells_Masters-SF-Aaron_Krick...,Aaron Krickstein,0.589286,0.848485,0.391304,0.035714,0.071429,0.245283,0.045872,0.146789,...,0.569495,0.600131,0.438864,0.048747,0.010323,0.353349,0.119122,0.153313,0.375443,0.505703
13830,19950122-M-Australian_Open-R16-Aaron_Krickstei...,Aaron Krickstein,0.610000,0.639344,0.564103,0.060000,0.005000,0.370968,0.163212,0.088083,...,0.572794,0.641524,0.430938,0.046575,0.020508,0.335338,0.106914,0.152226,0.412869,0.476975
13824,19950127-M-Australian_Open-SF-Aaron_Krickstein...,Aaron Krickstein,0.533333,0.562500,0.500000,0.050000,0.000000,0.264151,0.079646,0.141593,...,0.578109,0.641212,0.449961,0.048493,0.018292,0.340428,0.114956,0.143062,0.408310,0.504074
4069,20230219-M-Manama_CH-F-Thanasi_Kokkinakis-Abed...,Abedallah Shelbayh,0.619048,0.512821,0.583333,0.015873,0.015873,0.271186,0.180328,0.188525,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4063,20230221-M-Doha-R32-Soon_Woo_Kwon-Abedallah_Sh...,Abedallah Shelbayh,0.574713,0.740000,0.486486,0.091954,0.022989,0.344828,0.172414,0.206897,...,0.619048,0.512821,0.583333,0.015873,0.015873,0.271186,0.180328,0.188525,0.727273,0.666667


##### Extraindo informações de KeyPoints

In [22]:
# Calcular ratios
kps['bp_clutch_save_pct']    = kps['pts_won']    / kps['pts'].replace(0, np.nan)
kps['bp_first_in_pct']       = kps['first_in']   / kps['pts'].replace(0, np.nan)

kpr['bpo_conv_pct']          = kpr['pts_won']    / kpr['pts'].replace(0, np.nan)
kpr['bpo_ue_pct']            = kpr['unforced']   / kpr['pts'].replace(0, np.nan)

# Adicionar data
kps = kps.merge(matches[['match_id', 'Date']], on='match_id', how='left')
kpr = kpr.merge(matches[['match_id', 'Date']], on='match_id', how='left')

kps['Date'] = pd.to_datetime(kps['Date'], format='%Y%m%d', errors='coerce')
kpr['Date'] = pd.to_datetime(kpr['Date'], format='%Y%m%d', errors='coerce')

print(kps.shape, kpr.shape)


(60472, 15) (60472, 11)


In [23]:
kps = kps.sort_values(['player', 'Date'])
kpr = kpr.sort_values(['player', 'Date'])

for col in ['bp_clutch_save_pct', 'bp_first_in_pct']:
    kps[f'avg_{col}'] = (
        kps.groupby('player')[col]
        .transform(lambda x: x.expanding().mean().shift(1))
    )

for col in ['bpo_conv_pct', 'bpo_ue_pct']:
    kpr[f'avg_{col}'] = (
        kpr.groupby('player')[col]
        .transform(lambda x: x.expanding().mean().shift(1))
    )

kps_avg = kps[['match_id', 'player', 'avg_bp_clutch_save_pct', 'avg_bp_first_in_pct']]
kpr_avg = kpr[['match_id', 'player', 'avg_bpo_conv_pct', 'avg_bpo_ue_pct']]

print(kps_avg.shape, kpr_avg.shape)


(60472, 4) (60472, 4)


In [24]:
kps_agg = (kps.groupby(['match_id', 'player', 'Date'], as_index=False)
              [['pts', 'pts_won', 'first_in']]
              .sum())

kpr_agg = (kpr.groupby(['match_id', 'player', 'Date'], as_index=False)
              [['pts', 'pts_won', 'unforced']]
              .sum())

kps_agg['bp_clutch_save_pct'] = kps_agg['pts_won'] / kps_agg['pts'].replace(0, np.nan)
kps_agg['bp_first_in_pct']    = kps_agg['first_in'] / kps_agg['pts'].replace(0, np.nan)

kpr_agg['bpo_conv_pct'] = kpr_agg['pts_won'] / kpr_agg['pts'].replace(0, np.nan)
kpr_agg['bpo_ue_pct']   = kpr_agg['unforced'] / kpr_agg['pts'].replace(0, np.nan)

kps_agg = kps_agg.sort_values(['player', 'Date'])
kpr_agg = kpr_agg.sort_values(['player', 'Date'])

for col in ['bp_clutch_save_pct', 'bp_first_in_pct']:
    kps_agg[f'avg_{col}'] = (
        kps_agg.groupby('player')[col]
        .transform(lambda x: x.expanding().mean().shift(1))
    )

for col in ['bpo_conv_pct', 'bpo_ue_pct']:
    kpr_agg[f'avg_{col}'] = (
        kpr_agg.groupby('player')[col]
        .transform(lambda x: x.expanding().mean().shift(1))
    )

kps_avg = kps_agg[['match_id', 'player', 'avg_bp_clutch_save_pct', 'avg_bp_first_in_pct']]
kpr_avg = kpr_agg[['match_id', 'player', 'avg_bpo_conv_pct', 'avg_bpo_ue_pct']]

print(kps_avg.shape, kpr_avg.shape)


(15090, 4) (15090, 4)


##### Extraindo informações de Rally

In [25]:
rally.head()

,match_id,server,returner,row,pts,pl1_won,pl1_winners,pl1_forced,pl1_unforced,pl2_won,pl2_winners,pl2_forced,pl2_unforced
0,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,Jesper De Jong,Michael Zheng,Total,141,64,24,19,21,77,28,24,19
1,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,Jesper De Jong,Michael Zheng,1-3,72,37,10,16,11,35,10,10,9
2,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,Jesper De Jong,Michael Zheng,4-6,30,17,9,2,2,13,3,8,6
3,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,Jesper De Jong,Michael Zheng,7-9,25,7,4,1,4,18,8,6,2
4,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,Jesper De Jong,Michael Zheng,10,14,3,1,0,4,11,7,0,2


In [26]:
rally_filt = rally[rally['row'].isin(['1-3', '4-6', '7-9', '10'])].copy()
rally_filt = rally_filt.merge(matches[['match_id', 'Date']], on='match_id', how='left')
rally_filt['Date'] = pd.to_datetime(rally_filt['Date'], format='%Y%m%d', errors='coerce')

rally_filt['serve_win_pct']  = rally_filt['pl1_won'] / rally_filt['pts'].replace(0, np.nan)
rally_filt['return_win_pct'] = rally_filt['pl2_won'] / rally_filt['pts'].replace(0, np.nan)

rally_filt['row'] = rally_filt['row'].replace({'7-9': '7plus', '10': '7plus'})

print(rally_filt['row'].value_counts())
print(rally_filt.shape)

row
7plus    14924
1-3       7559
4-6       7559
Name: count, dtype: int64
(30042, 16)


In [27]:
# Agrupar 7plus somando os pontos
rally_agg = rally_filt.groupby(['match_id', 'server', 'returner', 'row', 'Date'], as_index=False)[
    ['pts', 'pl1_won', 'pl2_won']
].sum()

# Agora calcular win rate
rally_agg['serve_win_pct']  = rally_agg['pl1_won'] / rally_agg['pts'].replace(0, np.nan)
rally_agg['return_win_pct'] = rally_agg['pl2_won'] / rally_agg['pts'].replace(0, np.nan)

print(rally_agg['row'].value_counts())
print(rally_agg.shape)


row
1-3      7545
4-6      7545
7plus    7538
Name: count, dtype: int64
(22628, 10)


In [28]:
# Perspectiva de quem serve
serve_df = rally_agg[['match_id', 'server', 'row', 'Date', 'serve_win_pct']].copy()
serve_df = serve_df.rename(columns={'server': 'player'})
serve_df = serve_df.pivot_table(index=['match_id', 'player', 'Date'],
                                 columns='row', values='serve_win_pct').reset_index()
serve_df.columns = ['match_id', 'player', 'Date',
                    'srv_win_1_3', 'srv_win_4_6', 'srv_win_7plus']

# Perspectiva de quem retorna
return_df = rally_agg[['match_id', 'returner', 'row', 'Date', 'return_win_pct']].copy()
return_df = return_df.rename(columns={'returner': 'player'})
return_df = return_df.pivot_table(index=['match_id', 'player', 'Date'],
                                   columns='row', values='return_win_pct').reset_index()
return_df.columns = ['match_id', 'player', 'Date',
                     'ret_win_1_3', 'ret_win_4_6', 'ret_win_7plus']

rally_by_player = serve_df.merge(return_df, on=['match_id', 'player', 'Date'], how='outer')

print(rally_by_player.shape)
rally_by_player.head(3)


(15090, 9)


,match_id,player,Date,srv_win_1_3,srv_win_4_6,srv_win_7plus,ret_win_1_3,ret_win_4_6,ret_win_7plus
0,19600529-M-Roland_Garros-F-Nicola_Pietrangeli-...,Luis Ayala,1960-05-29,NaN,NaN,NaN,0.459016,0.510204,0.517241
1,19600529-M-Roland_Garros-F-Nicola_Pietrangeli-...,Nicola Pietrangeli,1960-05-29,0.540984,0.489796,0.482759,NaN,NaN,NaN
2,19600704-M-Wimbledon-F-Rod_Laver-Neale_Fraser,Neale Fraser,1960-07-04,NaN,NaN,NaN,0.523179,0.540541,0.500000


In [29]:
rally_by_player = rally_by_player.sort_values(['player', 'Date'])

rally_cols = ['srv_win_1_3', 'srv_win_4_6', 'srv_win_7plus',
              'ret_win_1_3', 'ret_win_4_6', 'ret_win_7plus']

for col in rally_cols:
    rally_by_player[f'avg_{col}'] = (
        rally_by_player.groupby('player')[col]
        .transform(lambda x: x.expanding().mean().shift(1))
    )

rally_avg = rally_by_player[['match_id', 'player'] + [f'avg_{c}' for c in rally_cols]]

print(rally_avg.shape)


(15090, 8)


##### Criando o dataset final dos perfis dos jogadores

In [30]:
ov_avg = ov_ratios[['match_id', 'player'] + [f'avg_{c}' for c in ratio_feature_cols]]

player_profiles = (ov_avg
    .merge(kps_avg,   on=['match_id', 'player'], how='left')
    .merge(kpr_avg,   on=['match_id', 'player'], how='left')
    .merge(rally_avg, on=['match_id', 'player'], how='left'))

print(player_profiles.shape)
player_profiles.head(3)


(15118, 22)


,match_id,player,avg_first_serve_pct,avg_first_serve_won_pct,avg_second_serve_won_pct,avg_ace_pct,avg_df_pct,avg_return_won_pct,avg_winners_per_pt,avg_ue_per_pt,...,avg_bp_clutch_save_pct,avg_bp_first_in_pct,avg_bpo_conv_pct,avg_bpo_ue_pct,avg_srv_win_1_3,avg_srv_win_4_6,avg_srv_win_7plus,avg_ret_win_1_3,avg_ret_win_4_6,avg_ret_win_7plus
0,19890909-M-US_Open-SF-Boris_Becker-Aaron_Krick...,Aaron Krickstein,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,19900415-M-Tokyo_Outdoor-F-Aaron_Krickstein-St...,Aaron Krickstein,0.447917,0.720930,0.377358,0.062500,0.041667,0.414414,0.149758,0.149758,...,0.476190,0.452381,0.345455,0.109091,NaN,NaN,NaN,0.447619,0.44898,0.528302
2,19910902-M-US_Open-R16-Aaron_Krickstein-Jimmy_...,Aaron Krickstein,0.576522,0.669556,0.362592,0.050481,0.020833,0.401652,0.151546,0.141546,...,0.509524,0.554762,0.382405,0.086804,0.439024,0.454545,0.542857,0.447619,0.44898,0.528302


In [31]:
print(player_profiles.duplicated(['match_id','player']).sum())

player_profiles = player_profiles.drop_duplicates(['match_id','player'])
print(player_profiles.shape)


28
(15090, 22)


##### Criando o Elo

In [32]:
matches_sorted = matches[['match_id', 'Player 1', 'Player 2', 'Date']].copy()
matches_sorted['Date'] = pd.to_datetime(matches_sorted['Date'], format='%Y%m%d', errors='coerce')
matches_sorted = matches_sorted.dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)

K = 32
elo_ratings = {}  # jogador → elo atual

elo_records = []  # guardar elo ANTES de cada partida

for _, row in matches_sorted.iterrows():
    p1, p2 = row['Player 1'], row['Player 2']

    elo1 = elo_ratings.get(p1, 1500)
    elo2 = elo_ratings.get(p2, 1500)

    # Guardar elo pré-partida
    elo_records.append({'match_id': row['match_id'], 'player': p1, 'elo': elo1})
    elo_records.append({'match_id': row['match_id'], 'player': p2, 'elo': elo2})

    # Atualizar elo (P1 sempre ganha)
    expected1 = 1 / (1 + 10 ** ((elo2 - elo1) / 400))
    expected2 = 1 - expected1

    elo_ratings[p1] = elo1 + K * (1 - expected1)
    elo_ratings[p2] = elo2 + K * (0 - expected2)

elo_df = pd.DataFrame(elo_records)
print(elo_df.shape)
elo_df.head(6)


(15130, 3)


,match_id,player,elo
0,19600529-M-Roland_Garros-F-Nicola_Pietrangeli-...,Nicola Pietrangeli,1500.0
1,19600529-M-Roland_Garros-F-Nicola_Pietrangeli-...,Luis Ayala,1500.0
2,19600704-M-Wimbledon-F-Rod_Laver-Neale_Fraser,Rod Laver,1500.0
3,19600704-M-Wimbledon-F-Rod_Laver-Neale_Fraser,Neale Fraser,1500.0
4,19690703-M-Wimbledon-SF-Rod_Laver-Arthur_Ashe,Rod Laver,1516.0
5,19690703-M-Wimbledon-SF-Rod_Laver-Arthur_Ashe,Arthur Ashe,1500.0


In [33]:
player_profiles = player_profiles.merge(elo_df, on=['match_id', 'player'], how='left')

print(player_profiles.shape)
print(player_profiles['elo'].isna().sum(), 'NaN no elo')


(15090, 23)
0 NaN no elo


##### Inserindo o ELO por superfície

In [34]:
matches_sorted = matches_sorted.merge(matches[['match_id', 'Surface']], on='match_id', how='left')

In [35]:
K = 32
elo_surface = {}  # (player, surface) → elo

surface_elo_records = []

for _, row in matches_sorted.iterrows():
    p1, p2, surf = row['Player 1'], row['Player 2'], row.get('Surface', np.nan)
    
    if pd.isna(surf):
        continue

    key1 = (p1, surf)
    key2 = (p2, surf)

    elo1 = elo_surface.get(key1, 1500)
    elo2 = elo_surface.get(key2, 1500)

    surface_elo_records.append({'match_id': row['match_id'], 'player': p1, 'elo_surface': elo1})
    surface_elo_records.append({'match_id': row['match_id'], 'player': p2, 'elo_surface': elo2})

    expected1 = 1 / (1 + 10 ** ((elo2 - elo1) / 400))
    elo_surface[key1] = elo1 + K * (1 - expected1)
    elo_surface[key2] = elo2 + K * (0 - (1 - expected1))

surface_elo_df = pd.DataFrame(surface_elo_records)
player_profiles = player_profiles.merge(surface_elo_df, on=['match_id', 'player'], how='left')

print(player_profiles.shape)
print(player_profiles['elo_surface'].isna().sum(), 'NaN no elo_surface')


(15092, 24)
0 NaN no elo_surface


In [36]:
player_profiles = player_profiles.drop_duplicates(['match_id', 'player'])
print(player_profiles.shape)


(15090, 24)


In [37]:
print(matches.duplicated('match_id').sum())


1


##### Montando o dataset final

In [38]:
# Separar perfil do P1 e P2
p1_profiles = player_profiles.merge(matches[['match_id', 'Player 1']], on='match_id', how='left')
p1_profiles = p1_profiles[p1_profiles['player'] == p1_profiles['Player 1']].drop(columns=['Player 1', 'player'])
p1_profiles.columns = ['match_id'] + [f'p1_{c}' for c in p1_profiles.columns if c != 'match_id']

p2_profiles = player_profiles.merge(matches[['match_id', 'Player 2']], on='match_id', how='left')
p2_profiles = p2_profiles[p2_profiles['player'] == p2_profiles['Player 2']].drop(columns=['Player 2', 'player'])
p2_profiles.columns = ['match_id'] + [f'p2_{c}' for c in p2_profiles.columns if c != 'match_id']

print(p1_profiles.shape, p2_profiles.shape)


(7545, 23) (7545, 23)


In [39]:
df_match = (matches[['match_id', 'Surface', 'Round', 'Best of']]
    .merge(p1_profiles, on='match_id', how='inner')
    .merge(p2_profiles, on='match_id', how='inner'))

df_match['target'] = 1  # Player 1 sempre ganha

print(df_match.shape)
df_match.head(3)


(7546, 49)


,match_id,Surface,Round,Best of,p1_avg_first_serve_pct,p1_avg_first_serve_won_pct,p1_avg_second_serve_won_pct,p1_avg_ace_pct,p1_avg_df_pct,p1_avg_return_won_pct,...,p2_avg_bpo_ue_pct,p2_avg_srv_win_1_3,p2_avg_srv_win_4_6,p2_avg_srv_win_7plus,p2_avg_ret_win_1_3,p2_avg_ret_win_4_6,p2_avg_ret_win_7plus,p2_elo,p2_elo_surface,target
0,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,Clay,Q3,3,0.589982,0.737376,0.518159,0.100131,0.032499,0.376990,...,0.173810,0.500507,0.525040,0.506261,0.550388,0.528302,0.357143,1515.195412,1500.000000,1
1,20260517-M-Rome_Masters-F-Casper_Ruud-Jannik_S...,Clay,F,3,0.656137,0.734247,0.549426,0.071281,0.025810,0.368573,...,0.149633,0.553612,0.545371,0.558828,0.531473,0.562615,0.559883,1427.659688,1473.710337,1
2,20260511-M-Rome_Masters-R16-Rafael_Jodar-Learn...,Clay,R16,3,0.626336,0.685951,0.485920,0.055664,0.018866,0.392246,...,0.127773,0.489626,0.495898,0.554170,0.484208,0.564322,0.541220,1445.226949,1500.000000,1


In [40]:
df_match = df_match.drop_duplicates('match_id')
print(df_match.shape)

(7545, 49)


##### Balanceando o dataset para não ter viés para o 1 jogador

In [41]:
df_mirror = df_match.copy()

# Trocar colunas p1 e p2
p1_cols = [c for c in df_match.columns if c.startswith('p1_')]
p2_cols = [c for c in df_match.columns if c.startswith('p2_')]

df_mirror[p1_cols] = df_match[p2_cols].values
df_mirror[p2_cols] = df_match[p1_cols].values
df_mirror['target'] = 0

df_final = pd.concat([df_match, df_mirror], ignore_index=True)

print(df_final.shape)
print(df_final['target'].value_counts())


(15090, 49)
target
1    7545
0    7545
Name: count, dtype: int64


In [47]:
import numpy as np

np.random.seed(42)
swap = np.random.rand(len(df_match)) > 0.5

df_final = df_match.copy()

p1_cols = [c for c in df_match.columns if c.startswith('p1_')]
p2_cols = [c for c in df_match.columns if c.startswith('p2_')]

df_final.loc[swap, p1_cols] = df_match.loc[swap, p2_cols].values
df_final.loc[swap, p2_cols] = df_match.loc[swap, p1_cols].values
df_final['target'] = (~pd.Series(swap)).astype(int).values

print(df_final.shape)
print(df_final['target'].value_counts())


(7545, 49)
target
1    3810
0    3735
Name: count, dtype: int64


In [45]:
df_final.shape

(15090, 49)

In [48]:
df_final.head()

,match_id,Surface,Round,Best of,p1_avg_first_serve_pct,p1_avg_first_serve_won_pct,p1_avg_second_serve_won_pct,p1_avg_ace_pct,p1_avg_df_pct,p1_avg_return_won_pct,...,p2_avg_bpo_ue_pct,p2_avg_srv_win_1_3,p2_avg_srv_win_4_6,p2_avg_srv_win_7plus,p2_avg_ret_win_1_3,p2_avg_ret_win_4_6,p2_avg_ret_win_7plus,p2_elo,p2_elo_surface,target
0,20260521-M-Roland_Garros-Q3-Jesper_De_Jong-Mic...,Clay,Q3,3,0.589982,0.737376,0.518159,0.100131,0.032499,0.376990,...,0.173810,0.500507,0.525040,0.506261,0.550388,0.528302,0.357143,1515.195412,1500.000000,1
1,20260517-M-Rome_Masters-F-Casper_Ruud-Jannik_S...,Clay,F,3,0.608739,0.771935,0.565165,0.088339,0.025925,0.412684,...,0.132300,0.528798,0.520978,0.512327,0.496060,0.523575,0.518140,1799.903684,1780.598904,0
2,20260511-M-Rome_Masters-R16-Rafael_Jodar-Learn...,Clay,R16,3,0.637767,0.696682,0.530308,0.063511,0.042448,0.385287,...,0.155412,0.513882,0.476432,0.498129,0.508333,0.564103,0.450000,1580.622731,1533.864955,0
3,20260507-M-Rome_Masters-R128-Stefanos_Tsitsipa...,Clay,R128,3,0.616676,0.701415,0.530257,0.073579,0.034647,0.352783,...,0.147681,0.523990,0.505157,0.500966,0.504397,0.493356,0.463498,1730.262256,1698.984943,0
4,20260503-M-Madrid_Masters-F-Jannik_Sinner-Alex...,Clay,F,3,0.608282,0.771396,0.565386,0.087943,0.025926,0.412243,...,0.154527,0.529369,0.498125,0.484769,0.511992,0.520992,0.497493,1464.977173,1416.902300,1


In [49]:
df_final.to_csv(f'{BASE}/tennis_final_dataset.csv', index=False)
print('Salvo com sucesso.')


Salvo com sucesso.


In [50]:
player_profiles.to_csv(f'{BASE}/tennis_player_profiles.csv', index=False)
print('Salvo.')

Salvo.


In [51]:
player_profiles.head()

,match_id,player,avg_first_serve_pct,avg_first_serve_won_pct,avg_second_serve_won_pct,avg_ace_pct,avg_df_pct,avg_return_won_pct,avg_winners_per_pt,avg_ue_per_pt,...,avg_bpo_conv_pct,avg_bpo_ue_pct,avg_srv_win_1_3,avg_srv_win_4_6,avg_srv_win_7plus,avg_ret_win_1_3,avg_ret_win_4_6,avg_ret_win_7plus,elo,elo_surface
0,19890909-M-US_Open-SF-Boris_Becker-Aaron_Krick...,Aaron Krickstein,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1500.000000,1500.000000
1,19900415-M-Tokyo_Outdoor-F-Aaron_Krickstein-St...,Aaron Krickstein,0.447917,0.720930,0.377358,0.062500,0.041667,0.414414,0.149758,0.149758,...,0.345455,0.109091,NaN,NaN,NaN,0.447619,0.44898,0.528302,1483.576959,1484.198785
2,19910902-M-US_Open-R16-Aaron_Krickstein-Jimmy_...,Aaron Krickstein,0.576522,0.669556,0.362592,0.050481,0.020833,0.401652,0.151546,0.141546,...,0.382405,0.086804,0.439024,0.454545,0.542857,0.447619,0.44898,0.528302,1496.696149,1496.914455
3,19911026-M-Stockholm_Masters-SF-Aaron_Krickste...,Aaron Krickstein,0.570086,0.666609,0.432739,0.045262,0.017206,0.396423,0.133289,0.140959,...,0.401995,0.087281,0.503200,0.425189,0.541799,0.447619,0.44898,0.528302,1510.175509,1515.506698
4,19920426-M-Monte_Carlo_Masters-F-Aaron_Krickst...,Aaron Krickstein,0.570422,0.633885,0.419793,0.054355,0.012904,0.363984,0.123903,0.134975,...,0.375026,0.080166,0.473398,0.341431,0.540686,0.447619,0.44898,0.528302,1519.882465,1500.000000


In [52]:
latest_profiles = player_profiles.sort_values('match_id').groupby('player').last().reset_index()
latest_profiles.to_csv(f'{BASE}/tennis_player_latest_profiles.csv', index=False)
print(latest_profiles.shape)

(1002, 24)
